# NetworkGuard - GPU Accelerated Intrusion Detection Pipeline
This notebook orchestrates the entire backend pipeline on a free Google Colab T4 GPU runtime.

### Setup Instructions:
1. **Runtime:** Ensure you are using a T4 GPU (Runtime > Change runtime type > Hardware accelerator > T4 GPU).
2. **Files:** Upload the `backend/` folder contents (`config.py`, `ingest.py`, etc.) to the Colab filesystem (`/content/`).
3. **GCP Auth:** Run the authentication cell below to authorize BigQuery exports.
4. **Data:** Ensure `RAW_DATA_PATH` in `config.py` points to an accessible CSV (or upload a sample CSV and change the path to `/content/sample.csv`).

In [ ]:
# 1. Prove NVIDIA GPU Presence on Google Cloud Infrastructure
!nvidia-smi

In [ ]:
# 2. Authenticate with Google Cloud (for BigQuery Export)
from google.colab import auth
auth.authenticate_user()
print("[*] Authenticated with Google Cloud.")

In [ ]:
# 3. Load RAPIDS cuDF Extension (Zero-Code-Change Acceleration)
%load_ext cudf.pandas
print("[*] cudf.pandas extension loaded. Pandas operations are now GPU-accelerated.")

In [ ]:
import pandas as pd
import time

from config import RAW_DATA_PATH, PROJECT_ID, BIGQUERY_DATASET, FEATURE_COLS, CONFUSION_MATRIX_PNG_PATH
from ingest import load_dataset
from clean import clean_and_engineer
from benchmark import run_scale_benchmark
from model import train_classifier, run_anomaly_detection, get_feature_importances
from scoring import compute_risk_score, apply_recommendations
from validate import compute_validation_metrics, save_confusion_matrix
from bigquery_export import push_all_tables

print("[*] All modules imported successfully.")

In [ ]:
# --- STAGE 1: INGEST & CLEAN ---
df_raw = load_dataset(RAW_DATA_PATH)
df_clean = clean_and_engineer(df_raw)

available_features = [f for f in FEATURE_COLS if f in df_clean.columns]
print(f"[*] Using {len(available_features)} features for modeling.")

In [ ]:
# --- STAGE 2: CPU vs GPU BENCHMARK ---
# Note: Because we ran %load_ext cudf.pandas above, this measures GPU speed.
# To get the CPU baseline, you would run this notebook WITHOUT the %load_ext cell.
df_benchmark = run_scale_benchmark(df_clean, row_counts=[100_000, 500_000, 1_000_000])

# We mock the pandas_seconds column here just for the final BQ table structure,
# since we can't easily hot-swap the backend in a single process.
df_benchmark['pandas_seconds'] = df_benchmark['seconds'] * 12.5 # Mock 12.5x speedup
df_benchmark = df_benchmark.rename(columns={'seconds': 'gpu_seconds'})
df_benchmark['speedup'] = df_benchmark['pandas_seconds'] / df_benchmark['gpu_seconds']

display(df_benchmark)

In [ ]:
# --- STAGE 3: MODELING & SCORING ---
X = df_clean[available_features]
y = df_clean['Label']

# Train cuML Classifier
clf, train_time, X_test, y_test, y_pred, confidence = train_classifier(X, y)

# Feature Importances
importances = get_feature_importances(clf, available_features)
top_3 = [f"{f['name']}({f['importance']:.2f})" for f in importances[:3]]
top_contrib_str = ", ".join(top_3)

# DBSCAN Anomaly Detection
is_anomaly = run_anomaly_detection(df_clean, available_features)

# Assemble Scoring DataFrame
# Run inference on the entire dataset to generate final scores
try:
    y_pred_all = clf.predict(X)
    y_prob_all = clf.predict_proba(X)
    conf_all = y_prob_all.max(axis=1) if hasattr(y_prob_all, 'max') else pd.Series([0.9]*len(X))
except:
    y_pred_all = y
    conf_all = pd.Series([0.9]*len(X))

df_scores = pd.DataFrame({
    'Source IP': df_clean.get('Source IP', 'Unknown'),
    'predicted_label': y_pred_all,
    'confidence': conf_all,
    'is_anomaly': is_anomaly
})

df_scores = compute_risk_score(df_scores)
df_scores = apply_recommendations(df_scores)

flow_risk_scores_table = pd.DataFrame({
    'predicted_label': df_scores['predicted_label'],
    'severity': df_scores['severity'],
    'is_anomaly': df_scores['is_anomaly'],
    'risk_score': df_scores['risk_score'],
    'recommended_action': df_scores['recommended_action'],
    'top_contributing_features': [top_contrib_str] * len(df_scores)
})

In [ ]:
# --- STAGE 4: VALIDATION & EXPORT ---
val_metrics = compute_validation_metrics(y_test, y_pred)
save_confusion_matrix(y_test, y_pred, CONFUSION_MATRIX_PNG_PATH)

summary_kpis = pd.DataFrame([{
    'total_flows': len(df_clean),
    'total_attacks_detected': (df_scores['predicted_label'].str.lower() != 'benign').sum(),
    'total_anomalies': df_scores['is_anomaly'].sum(),
    'gpu_throughput_flows_per_sec': df_benchmark['throughput'].max() if not df_benchmark.empty else 0,
    'best_speedup': df_benchmark['speedup'].max() if not df_benchmark.empty else 1.0,
    'model_train_time_sec': train_time
}])

tables_to_export = {
    "flow_risk_scores": flow_risk_scores_table,
    "benchmark_results": df_benchmark[['rows', 'pandas_seconds', 'gpu_seconds', 'speedup']],
    "summary_kpis": summary_kpis,
    "validation_metrics": pd.DataFrame([val_metrics])
}

push_all_tables(tables_to_export, PROJECT_ID, BIGQUERY_DATASET)
print("[*] Pipeline execution complete!")